In [1]:
import pathlib
import cv2
import numpy as np
import httpx
from dotenv import dotenv_values

In [2]:
env_path = pathlib.Path("../.env").resolve()
config = dotenv_values(env_path)

API_URL = config["AI_DEVS_API_URL"]
API_KEY = config["AI_DEVS_API_KEY"]

print(f"Loaded config. API_URL={API_URL}")

Loaded config. API_URL=https://hub.ag3nts.org


In [ ]:
def _crop_grid(img: np.ndarray) -> np.ndarray:
    h, w = img.shape[:2]
    x1, x2 = int(w * 0.28), int(w * 0.655)
    y1, y2 = int(h * 0.21), int(h * 0.87)
    return img[y1:y2, x1:x2]


def _ascii_art_converter(img: np.ndarray, width: int = 90) -> str:
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    _, binary = cv2.threshold(gray, 128, 255, cv2.THRESH_BINARY_INV)
    aspect_ratio = binary.shape[0] / binary.shape[1]
    height = int(width * aspect_ratio * 0.5)
    resized = cv2.resize(binary, (width, height), interpolation=cv2.INTER_AREA)
    ascii_chars = [" ", "+"]
    ascii_str = ""
    for row in resized:
        for pixel in row:
            ascii_str += ascii_chars[1] if pixel > 0 else ascii_chars[0]
        ascii_str += "\n"
    return ascii_str

resp = httpx.get(f"{API_URL}/data/{API_KEY}/electricity.png", timeout=20)
resp.raise_for_status()

arr = np.frombuffer(resp.content, dtype=np.uint8)
img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
grid = _crop_grid(img)
ascii_art = _ascii_art_converter(grid)

print(ascii_art)